# SEO Static HTML Sitemap Impact Analysis

 ## 1. Objective
Evaluate whether the supplementary static HTML sitemap, activated on March 13, 2026, increased Googlebot crawl frequency for the "problem pages" included in the sitemap compared with problem pages that were not included.

## Analysis groups
- **Treatment:** Problem pages from the 5 `_updated` category files included in the static HTML sitemap.
- **Control:** Problem pages from the 4 category files not included in the static HTML sitemap.
- **Intervention date:** March 13, 2026.
- ## Analysis periods
- **Before:** December 2, 2025 â€“ March 12, 2026
- **After:** March 13, 2026 â€“ March 29, 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Load and Prepare Problem Pages


In [ ]:
problem_pages_folder = Path.home() / "Downloads" / "seo_source_data" / "Problem Pages"

print(problem_pages_folder.exists())

In [ ]:
files = list(problem_pages_folder.glob("*.txt"))

print("Number of files:", len(files))

for file in files:
      print(file.name)               

In [ ]:
# an empty list to store all problem-page records
problem_pages = []

# Process each of the 9 root-category files
for file in files:
    category = file.stem.split("_")[1]

    # "_updated" files contain pages included in the static HTML sitemap
    if "_updated" in file.name:
        sitemap_group = "Included"
    else:
        sitemap_group = "Not Included"

    # Read each problem-page URL
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            url = line.strip()

             # Skip empty lines
            if not url:
                continue

            # Extract product_id from the page URL for joining with Googlebot data
            product_id = url.split("/confronta-prezzi/")[1].split("/")[0]

            problem_pages.append({
                "product_id": product_id,
                "category": category,
                "sitemap_group": sitemap_group,
                "url": url
            })

# Convert the collected records into a pandas DataFrame
problem_pages_df = pd.DataFrame(problem_pages)

In [ ]:
problem_pages_df = pd.DataFrame(problem_pages)

problem_pages_df.head() 

In [ ]:
problem_pages_df.shape

In [ ]:
problem_pages_df["sitemap_group"].value_counts()

In [ ]:
problem_pages_df.groupby(
    ["category", "sitemap_group"]
).size()

In [ ]:
problem_pages_df["product_id"].nunique()

In [ ]:
len(problem_pages_df)

## 3. Googlebot Crawl Data

Load and inspect the Googlebot request data covering December 2, 2025 to March 29, 2026.

In [ ]:
googlebot_file = Path.home() /"Downloads" / "seo_source_data" / "gb-pis_IT_OOP_sitemap.csv"/"gb-pis_IT_OOP_sitemap.csv"

print(googlebot_file.exists())

In [ ]:
googlebot_df = pd.read_csv(googlebot_file)

googlebot_df.head()

In [ ]:
googlebot_df.info()

# Correct invalid 29.02.2026 date to 28.02.2026.
# This does not affect Before/After classification, but may affect the daily trend around this date.

In [ ]:
bad_dates = pd.to_datetime(
    googlebot_df["date"],
    format="%d.%m.%Y",
    
    # if date is bad replace it with NaT (Not a Time)
    errors="coerce"
)

googlebot_df[bad_dates.isna()]["date"].value_counts()

In [ ]:
# Correct invalid source date: 29.02.2026 does not exist in 2026
googlebot_df["date"] = googlebot_df["date"].replace(
"29.02.2026",
"28.02.2026"
)

In [ ]:
googlebot_df["date"] = pd.to_datetime(
    googlebot_df["date"],
    format="%d.%m.%Y"
)

In [ ]:
googlebot_df["date"].min(),googlebot_df["date"].max()

In [ ]:
googlebot_df["indexstatus"].value_counts(dropna=False)

In [ ]:
googlebot_df["pis"].value_counts().head(10)

In [ ]:
googlebot_df.isna().sum()

In [ ]:
# Convert product_id to integer to match the Googlebot dataset
problem_pages_df["product_id"] = problem_pages_df["product_id"].astype("int64")


In [ ]:
problem_pages_df["product_id"].dtype

In [ ]:
# Keep only the problem-page product IDs so we can filter the large Googlebot dataset before merging
problem_ids = set(problem_pages_df["product_id"])

len(problem_ids)

In [ ]:
# Filter Googlebot data to keep only crawl records belonging to the 7,871 identified problem pages
problem_crawls_df = googlebot_df[
    googlebot_df["product_id"].isin(problem_ids)
].copy()


In [ ]:
problem_crawls_df.shape

## 4. Join Problem Pages with Crawl Data

In [ ]:
# Merge crawl records with problem-page information using product_id
analysis_df = problem_crawls_df.merge(
    problem_pages_df[["product_id", "category", "sitemap_group"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
analysis_df.shape


In [ ]:
analysis_df.head()

In [ ]:
# Label each crawl record as before or after the static HTML sitemap activation
analysis_df["period"] = np.where(
    analysis_df["date"] < pd.Timestamp("2026-03-13"),
    "Before",
    "After"
)

In [ ]:
analysis_df["period"].value_counts()

In [ ]:
analysis_df.head()

In [ ]:
analysis_df["period"].value_counts()


In [ ]:
analysis_df.groupby(["sitemap_group", "period"]).size()

## 5. Before vs After Crawl Analysis

In [ ]:
# Calculate total Googlebot requests for each sitemap group and analysis period
analysis_df.groupby(["sitemap_group", "period",])["pis"].sum()

In [ ]:
# Summarize total Googlebot requests for each sitemap group and analysis period
crawl_summary = (
    analysis_df
    .groupby(["sitemap_group", "period"])["pis"]
    .sum()
    .reset_index(name="total_crawl_requests")
)

crawl_summary

In [ ]:
# Count the number of unique problem pages in each sitemap group
page_counts = (
    problem_pages_df
    .groupby("sitemap_group")["product_id"]
    .nunique()
    .reset_index(name="page_count")
)

page_counts

In [ ]:
# # Add page counts to the crawl summary for normalization
crawl_summary = crawl_summary.merge(
    page_counts,
    on="sitemap_group",
    how="left"
)

crawl_summary

## 6. Daily Crawl Trend

In [ ]:
# Define the number of days in each analysis period
period_days = {
    "Before": 101,
    "After": 17
}

In [ ]:
# Add the number of days in each period for normalization
crawl_summary["days"] = crawl_summary["period"].map(period_days)

crawl_summary

In [ ]:
# Calculate normalized Googlebot crawl requests per page per day
crawl_summary["requests_per_page_per_day"] = (
    crawl_summary["total_crawl_requests"]
    / (crawl_summary["page_count"] * crawl_summary["days"])
)

crawl_summary

In [ ]:
# Reshape the normalized crawl rates for Before/After comparison
rate_comparison = crawl_summary.pivot(
    index="sitemap_group",
    columns="period",
    values="requests_per_page_per_day"
)

rate_comparison

In [ ]:
# Calculate the percentage change in crawl rate from Before to After
rate_comparison["pct_change"] = (
    (rate_comparison["After"] - rate_comparison["Before"])
    / rate_comparison["Before"]
) * 100

rate_comparison

In [ ]:
# Calculate total Googlebot crawl requests per day and sitemap group
daily_crawls = (
    analysis_df
    .groupby(["date", "sitemap_group"])["pis"]
    .sum()
    .reset_index(name="total_crawl_requests")
)

daily_crawls.head(10)

In [ ]:
# Add page counts to the daily crawl data for normalization
daily_crawls = daily_crawls.merge(
    page_counts,
    on="sitemap_group",
    how="left"
)

daily_crawls.head(10)

In [ ]:
# Calculate daily Googlebot crawl requests per page
daily_crawls["requests_per_page"] = (
    daily_crawls["total_crawl_requests"]
    / daily_crawls["page_count"]
)

daily_crawls.head(10)

In [ ]:
# Plot daily normalized crawl rates
plt.figure(figsize=(12, 6))

for group in ["Included", "Not Included"]:
    group_data = daily_crawls[daily_crawls["sitemap_group"] == group]

    plt.plot(
        group_data["date"],
        group_data["requests_per_page"],
        label=group
    )

plt.axvline(
    pd.Timestamp("2026-03-13"),
    linestyle="--",
    label="Sitemap activation"
)

plt.xlabel("Date")
plt.ylabel("Crawl requests per page")
plt.title("Daily Googlebot Crawl Rate: Included vs Not Included")
plt.legend()
plt.show()

In [ ]:
# Calculate a 7-day rolling average to smooth daily fluctuations
daily_crawls["rolling_7d"] = (
    daily_crawls
    .groupby("sitemap_group")["requests_per_page"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

In [ ]:
# Plot the 7-day rolling average of normalized crawl rates
plt.figure(figsize=(12, 6))

for group in ["Included", "Not Included"]:
    group_data = daily_crawls[daily_crawls["sitemap_group"] == group]

    plt.plot(
        group_data["date"],
        group_data["rolling_7d"],
        label=group
    )

plt.axvline(
    pd.Timestamp("2026-03-13"),
    linestyle="--",
    label="Sitemap activation"
)

plt.xlabel("Date")
plt.ylabel("7-day average crawl requests per page")
plt.title("7-Day Average Googlebot Crawl Rate: Included vs Not Included")
plt.legend()
plt.show()

## 7. Category-Level Analysis

In [ ]:
# Calculate total crawl requests for each category before and after activation
category_summary = (
    analysis_df
    .groupby(["category", "sitemap_group", "period"])["pis"]
    .sum()
    .reset_index(name="total_crawl_requests")
)

category_summary

In [ ]:
# Count the number of problem pages in each category
category_page_counts = (
    problem_pages_df
    .groupby("category")["product_id"]
    .nunique()
    .reset_index(name="page_count")
)

category_page_counts

In [ ]:
# Add page counts to the category summary for normalization
category_summary = category_summary.merge(
    category_page_counts,
    on="category",
    how="left"
)

category_summary

In [ ]:
# Add the number of days in each period
category_summary["days"] = category_summary["period"].map(period_days)

In [ ]:
# Calculate crawl requests per page per day for each category
category_summary["requests_per_page_per_day"] = (
    category_summary["total_crawl_requests"]
    / (category_summary["page_count"] * category_summary["days"])
)

category_summary

In [ ]:
# Reshape category crawl rates for Before/After comparison
category_comparison = category_summary.pivot(
    index=["category", "sitemap_group"],
    columns="period",
    values="requests_per_page_per_day"
)

category_comparison

In [ ]:
# Calculate the percentage change for each category
category_comparison["pct_change"] = (
    (category_comparison["After"] - category_comparison["Before"])
    / category_comparison["Before"]
) * 100

category_comparison

## 8. Matched 17-Day Robustness Check

In [ ]:
# Keep only the 17 days before and 17 days after sitemap activation
matched_df = analysis_df[
    analysis_df["date"] >= pd.Timestamp("2026-02-24")
].copy()

matched_df["matched_period"] = np.where(
    matched_df["date"] < pd.Timestamp("2026-03-13"),
    "Before",
    "After"
)

matched_df["matched_period"].value_counts()

In [ ]:
# Calculate total crawl requests for the matched 17-day periods
matched_summary = (
    matched_df
    .groupby(["sitemap_group", "matched_period"])["pis"]
    .sum()
    .reset_index(name="total_crawl_requests")
)

matched_summary

In [ ]:
# Add page counts to the matched-period summary
matched_summary = matched_summary.merge(
    page_counts,
    on="sitemap_group",
    how="left"
)

# Calculate crawl requests per page per day
matched_summary["requests_per_page_per_day"] = (
    matched_summary["total_crawl_requests"]
    / (matched_summary["page_count"] * 17)
)

matched_summary

In [ ]:
# Compare Before and After crawl rates in the matched 17-day periods
matched_comparison = matched_summary.pivot(
    index="sitemap_group",
    columns="matched_period",
    values="requests_per_page_per_day"
)

matched_comparison["pct_change"] = (
    (matched_comparison["After"] - matched_comparison["Before"])
    / matched_comparison["Before"]
) * 100

matched_comparison

## 9. Meta-Robots Status Check

In [ ]:
# Compare Googlebot crawl requests by sitemap group, period, and meta-robots status
indexstatus_summary = (
    analysis_df
    .groupby(["sitemap_group", "period", "indexstatus"])["pis"]
    .sum()
    .reset_index(name="total_crawl_requests")
)

indexstatus_summary

In [ ]:
# Calculate the share of crawl requests where Googlebot encountered an index status
index_share = (
    indexstatus_summary
    .pivot_table(
        index=["sitemap_group", "period"],
        columns="indexstatus",
        values="total_crawl_requests"
    )
    .reset_index()
)

index_share["index_share_pct"] = (
    index_share["index"]
    / (index_share["index"] + index_share["noindex"])
) * 100

index_share

The share of crawl requests encountering `index` increased from 56.7% to 72.6% for Included pages and from 53.2% to 77.4% for Not Included pages.

## 10. Conclusion

The analysis does not show evidence that the supplementary static HTML sitemap improved Googlebot crawl frequency during the observed post-activation period.

### Key findings

- Across the full analysis period, normalized crawl frequency decreased by approximately 49.95% for pages included in the static HTML sitemap, compared with a 14.86% decrease for pages not included.
- The daily trend shows that crawl frequency for the Included group had already started declining before the sitemap activation on March 13.
- All five Included root categories experienced a decline in normalized crawl frequency, indicating that the overall result was not driven by a single category.
- In the matched 17-day comparison, Included pages decreased by approximately 27.82%, while Not Included pages remained nearly stable at -0.61%.
- The share of crawl requests encountering `index` increased for both groups: from 56.7% to 72.6% for Included pages and from 53.2% to 77.4% for Not Included pages. Because the increase was larger for Not Included pages, this does not indicate a sitemap-specific improvement.

### Interpretation

Based on the available data, there is no clear evidence that the static HTML sitemap produced a positive effect on Googlebot crawl frequency during the 17-day post-activation period.

The meta-robots results also do not show a sitemap-specific improvement. Although Googlebot encountered `index` more frequently after activation, this increase occurred in both groups and was larger among pages that were not included in the sitemap.

However, the results should not be interpreted as evidence that the sitemap caused the decline in crawl frequency. The Included pages were already trending downward before activation, the categories were not randomly assigned, and only 17 days of post-activation data are available.